<a href="https://colab.research.google.com/github/Artty02/Assignment/blob/main/mocoipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# STEP 0 : ติดตั้งไลบรารีหลัก + เมานต์ Drive
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip -q install pycocotools albumentations tqdm

from google.colab import drive
drive.mount('/content/drive')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 86.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 54.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 125.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 41.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 16.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.7/848.7 MB 1.9 MB/s eta 0:00:00


MessageError: Error: credential propagation was unsuccessful

In [ ]:
# STEP 1 : path / config
from pathlib import Path
import yaml, json, torch, torchvision

ROOT_TACO = Path("/content/drive/MyDrive/TACO/taco_yolo")
UNLAB_ROOT = Path("/content/drive/MyDrive/MoCo")       # ภาพ unlabeled
SAVE_DIR   = Path("/content/drive/MyDrive/MoCo_ckpt")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ---------- เตรียม category_map จาก data.yaml ----------
with open(ROOT_TACO / "data.yaml") as f:
    names = yaml.safe_load(f)["names"]        # 18 รายการ
category_map = {i: n for i, n in enumerate(names)}
NUM_CLASSES  = len(category_map) + 1          # + background

print("✓ category_map", len(category_map), "classes")


In [ ]:
# STEP 2 : โมเดล MoCo V2 (ย่อจากไฟล์ของอาร์ต)
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T

class MoCoV2Augmentation:
    def __init__(self, size=224):
        self.aug = T.Compose([
            T.RandomResizedCrop(size, scale=(0.2, 1)),
            T.RandomApply([T.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8),
            T.RandomGrayscale(0.2),
            T.RandomApply([T.GaussianBlur(23, sigma=(0.1,2.0))], p=0.5),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    def __call__(self, x):
        return self.aug(x), self.aug(x)

class MoCoV2ForObjectDetection(nn.Module):
    def __init__(self, base_encoder, dim=256, K=65536, m=0.999, T=0.07):
        super().__init__()
        self.K, self.m, self.T = K, m, T
        # encoders
        self.encoder_q = base_encoder(num_classes=dim)
        self.encoder_k = base_encoder(num_classes=dim)
        # MLP-head
        d = self.encoder_q.fc.weight.shape[1]
        mlp = nn.Sequential(nn.Linear(d,d), nn.ReLU(), self.encoder_q.fc)
        self.encoder_q.fc, self.encoder_k.fc = mlp, copy.deepcopy(mlp)
        # init k
        for p_q, p_k in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            p_k.data.copy_(p_q.data); p_k.requires_grad = False
        # queue
        self.register_buffer("queue", torch.randn(dim, K))
        self.queue = F.normalize(self.queue, dim=0)
        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def _momentum_update_key_encoder(self):
        for p_q, p_k in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            p_k.data = p_k.data * self.m + p_q.data * (1. - self.m)

    @torch.no_grad()
    def _dequeue_and_enqueue(self, keys):
        b = keys.shape[0]; ptr = int(self.queue_ptr)
        self.queue[:, ptr:ptr+b] = keys.T
        self.queue_ptr[0] = (ptr+b) % self.K

    def forward(self, im_q, im_k):
        q = F.normalize(self.encoder_q(im_q), dim=1)
        with torch.no_grad():
            self._momentum_update_key_encoder()
            k = F.normalize(self.encoder_k(im_k), dim=1)
        l_pos = torch.einsum('nc,nc->n', [q,k]).unsqueeze(1)            # Nx1
        l_neg = torch.einsum('nc,ck->nk', [q, self.queue.clone().detach()])
        logits = torch.cat([l_pos,l_neg], 1) / self.T
        labels = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
        self._dequeue_and_enqueue(k)
        return logits, labels


In [ ]:
# STEP 3 : unlabeled dataset
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import copy

moco_aug = MoCoV2Augmentation(224)
class MoCoFolder(ImageFolder):
    def __getitem__(self, idx):
        path,_ = self.samples[idx]; img = self.loader(path)
        return moco_aug(img)          # (q,k)

unlab_ds = MoCoFolder(UNLAB_ROOT)     # ต้องมี sub-folder/unknown/...
unlab_loader = DataLoader(
    unlab_ds, batch_size=128, shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True)
print("✓ unlabeled images:", len(unlab_ds))


In [ ]:
# STEP 4 : pretrain loop
import torchvision.models as models, torch, numpy as np, tqdm, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MoCoV2ForObjectDetection(models.resnet50).to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.02, momentum=0.9, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)

def accuracy(out, tgt, topk=(1,)):
    _, pred = out.topk(max(topk), 1, True, True); correct = pred.eq(tgt.view(-1,1))
    return [ correct[:,:k].flatten().float().sum()*100/out.size(0) for k in topk ]

best1 = 0
for epoch in range(100):
    model.train(); tids = time.time()
    loss_meter, acc_meter = [], []
    for (q,k) in tqdm.tqdm(unlab_loader, desc=f"E{epoch:03d}"):
        im_q = q.to(device, non_blocking=True)
        im_k = k.to(device, non_blocking=True)
        out, tgt = model(im_q, im_k)
        loss = F.cross_entropy(out, tgt)
        opt.zero_grad(); loss.backward(); opt.step()
        acc1 = accuracy(out, tgt, (1,))[0]
        loss_meter.append(loss.item()); acc_meter.append(acc1.item())
    sched.step()
    print(f"[E{epoch}] loss {np.mean(loss_meter):.4f} | acc@1 {np.mean(acc_meter):.2f} | {time.time()-tids:.1f}s")
    if np.mean(acc_meter) > best1:
        best1 = np.mean(acc_meter)
        torch.save(model.state_dict(), SAVE_DIR/"moco_v2_best.pth")
print("✓ Pretrain done – best acc@1", best1)


In [ ]:
# STEP 6 : สร้าง Faster R-CNN และย้ายน้ำหนัก
from torchvision.models.detection import fasterrcnn_resnet50_fpn

det_model = fasterrcnn_resnet50_fpn(pretrained=False, num_classes=NUM_CLASSES)
# --- load MoCo weight เข้า backbone ---
moco_state = torch.load(SAVE_DIR/"moco_v2_best.pth", map_location="cpu")
new_sd = {k.replace("encoder_q.",""):v for k,v in moco_state.items()
          if k.startswith("encoder_q") and not k.startswith("encoder_q.fc")}
msg = det_model.backbone.body.load_state_dict(new_sd, strict=False)
print("backbone load msg:", msg)

det_model = det_model.to(device)
